# Challenge 5: Clustering de Series de Tiempo de Ventas de Energía

**Objetivo:** Este notebook explora diferentes algoritmos de clustering no supervisado para agrupar series de tiempo de ventas de energía del sector minorista de la EIA (U.S. Energy Information Administration).

El análisis se realiza a nivel de `estado × mes`. Se aplicarán y compararán tres algoritmos de clustering principales:
1.  **K-Means**
2.  **DBSCAN**
3.  **Clustering Jerárquico Aglomerativo**

El objetivo es identificar patrones y segmentos naturales en los datos de ventas de energía, basados en características de ingeniería (features) que capturan tendencias, estacionalidad y volatilidad.

**Metodología:**
1.  **Carga y Preprocesamiento:** Cargar los datos, limpiar, realizar ingeniería de características y reducir la dimensionalidad con PCA.
2.  **Modelado de Clustering:** Aplicar K-Means, DBSCAN y Clustering Jerárquico.
3.  **Evaluación:** Evaluar la calidad de los clusters utilizando métricas como el Coeficiente de Silueta, Davies-Bouldin y Calinski-Harabasz.
4.  **Visualización y Perfilado:** Visualizar los clusters en 2D y analizar sus perfiles para interpretar los resultados.
5.  **Comparación:** Comparar el rendimiento de los diferentes algoritmos.

## 1. Importación de Librerías y Configuración Inicial

En esta celda, importamos todas las librerías necesarias para el análisis.
- `pandas` y `numpy` para manipulación de datos.
- `matplotlib` para visualizaciones.
- `sklearn` para los algoritmos de clustering (KMeans, DBSCAN, AgglomerativeClustering), preprocesamiento (StandardScaler), reducción de dimensionalidad (PCA) y métricas de evaluación.
- `scipy` para el clustering jerárquico.

También se definen constantes como la semilla aleatoria (`SEED`) para la reproducibilidad y se configuran los directorios de salida y el logging.

In [1]:
! pip install -r requirements.txt


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from __future__ import annotations

import os
import logging
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")   # headless — no display needed
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    silhouette_score, silhouette_samples,
    davies_bouldin_score,
    calinski_harabasz_score,
)
from scipy.cluster.hierarchy import dendrogram, linkage
SEED = 42
np.random.seed(SEED)
CACHE_FILE = "eia_retail_sales.csv"
os.makedirs("figures", exist_ok=True)
os.makedirs("results", exist_ok=True)
os.makedirs("logs",    exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler("logs/challenge5.log", mode="w", encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
log = logging.getLogger(__name__)

## 2. Carga y Preprocesamiento de Datos

La función `load_and_preprocess` se encarga de:
1.  **Cargar los datos** desde el archivo `eia_retail_sales.csv`.
2.  **Filtrar** para mantener solo el sector agregado `'ALL'`.
3.  **Realizar ingeniería de características (feature engineering)** para crear variables relevantes para el clustering, como:
    *   Valores rezagados (`lag1`, `lag12`).
    *   Estadísticas móviles (`roll12_mean`, `roll12_std`).
    *   Crecimiento interanual (`yoy_growth`).
    *   Codificación de estacionalidad (usando seno y coseno del mes).
    *   Una tendencia lineal.
4.  **Limpiar** los datos eliminando filas con valores nulos.
5.  **Escalar las características** usando `StandardScaler` para que todas tengan una media de 0 y una desviación estándar de 1.
6.  **Aplicar Análisis de Componentes Principales (PCA)** para reducir la dimensionalidad, conservando el 90% de la varianza explicada. Esto ayuda a mitigar la "maldición de la dimensionalidad" y a mejorar el rendimiento de los algoritmos de clustering.

In [3]:

def load_and_preprocess() -> tuple[np.ndarray, pd.DataFrame]:
    """
    Loads EIA retail sales CSV and builds a feature matrix suitable for
    clustering.

    Unit of analysis: one row = one (state × month) observation using
    only the 'ALL sectors' aggregate (same aggregation as Challenge 2).

    Features engineered (all lag-based → no look-ahead):
      sales_raw       : raw monthly sales (MWh)
      price           : retail price (¢/kWh)
      revenue_per_mwh : revenue / sales — a normalised price proxy
      lag1, lag12     : 1-month and 12-month lagged sales
      roll12_mean     : 12-month trailing average
      roll12_std      : 12-month trailing std (volatility)
      yoy_growth      : year-over-year % change
      month_sin/cos   : Fourier seasonality encoding
      year_trend      : linear time index
      state_enc       : ordinal state code (for RF; numeric for clustering)

    Missing values: rows with any NaN after feature creation are dropped.
    """
    log.info(f"Loading data from {CACHE_FILE}")
    df_raw = pd.read_csv(CACHE_FILE)
    log.info(f"  Raw rows: {len(df_raw):,}")

    # Keep only ALL-sectors aggregate rows
    df = df_raw[df_raw["sectorid"] == "ALL"].copy()
    for col in ["sales", "price", "revenue", "customers"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["period"] = pd.to_datetime(df["period"])
    df = df.sort_values(["stateid", "period"]).reset_index(drop=True)
    df = df.dropna(subset=["sales"]).reset_index(drop=True)

    log.info(f"  After ALL-sector filter: {len(df):,} rows | "
             f"{df['stateid'].nunique()} states | "
             f"{df['period'].min().date()} → {df['period'].max().date()}")

    # ── Feature engineering ────────────────────────────────────────────
    grp = df.groupby("stateid")

    df["lag1"]        = grp["sales"].shift(1)
    df["lag12"]       = grp["sales"].shift(12)

    _past = grp["sales"].transform(lambda x: x.shift(1))
    df["roll12_mean"] = _past.rolling(12, min_periods=6).mean()
    df["roll12_std"]  = _past.rolling(12, min_periods=6).std()

    df["yoy_growth"]  = (
        (df["sales"] - df["lag12"]) / (df["lag12"].replace(0, np.nan) + 1e-9)
    ).clip(-5, 5)

    # Revenue per MWh (normalised price proxy; avoid div-by-zero)
    df["revenue_per_mwh"] = (
        df["revenue"] / (df["sales"].replace(0, np.nan) + 1e-9)
    )

    df["month_sin"] = np.sin(2 * np.pi * df["period"].dt.month / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["period"].dt.month / 12)
    df["year_trend"] = (
        (df["period"].dt.year - 2001) * 12 + df["period"].dt.month
    )
    df["state_enc"] = df["stateid"].astype("category").cat.codes

    FEATURES = [
    "yoy_growth",
    "roll12_std",
    "roll12_mean",
    "price",
    "month_sin",
    "month_cos",
    "year_trend",
    "revenue_per_mwh",
]
    df = df.dropna(subset=FEATURES).reset_index(drop=True)
    X_raw = df[FEATURES].values
    log.info(f"  Final dataset: {X_raw.shape[0]:,} rows × {X_raw.shape[1]} features")

    # ── Scale ──────────────────────────────────────────────────────────
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_raw)

    # ── PCA (retain ≥90% variance) ─────────────────────────────────────
    pca_full = PCA(random_state=SEED)
    pca_full.fit(X_scaled)
    cumvar = np.cumsum(pca_full.explained_variance_ratio_)
    n_comp = int(np.searchsorted(cumvar, 0.90)) + 1
    log.info(f"  PCA: {n_comp} components retain "
             f"{cumvar[n_comp-1]*100:.1f}% variance")

    pca = PCA(n_components=n_comp, random_state=SEED)
    X_pca = pca.fit_transform(X_scaled)

    # Save variance plot
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(np.arange(1, len(cumvar)+1), cumvar * 100, marker="o", ms=4)
    ax.axhline(90, color="red", ls="--", label="90% threshold")
    ax.axvline(n_comp, color="orange", ls="--", label=f"n={n_comp}")
    ax.set_xlabel("Number of PCA components")
    ax.set_ylabel("Cumulative explained variance (%)")
    ax.set_title("PCA — Cumulative Explained Variance")
    ax.legend()
    fig.tight_layout()
    fig.savefig("figures/00_pca_variance.png", dpi=120)
    plt.close(fig)

    return X_scaled, X_pca, df, FEATURES, pca

## 3. Métricas de Evaluación de Clustering

La función `compute_metrics` calcula tres métricas comunes para evaluar la calidad de los clusters:
-   **Coeficiente de Silueta (Silhouette Score):** Mide qué tan similar es un objeto a su propio cluster en comparación con otros clusters. Un valor alto indica clusters densos y bien separados. El rango es de -1 a 1.
-   **Índice de Davies-Bouldin (Davies-Bouldin Score):** Mide la similitud promedio entre cada cluster y su cluster más similar. Un valor bajo indica una mejor separación. El valor mínimo es 0.
-   **Índice de Calinski-Harabasz (Calinski-Harabasz Score):** También conocido como el criterio de Ratio de Varianza. Es la relación entre la dispersión entre clusters y la dispersión intra-cluster. Un valor más alto indica clusters mejor definidos.

Estas métricas nos ayudarán a comparar objetivamente los resultados de los diferentes algoritmos.

In [4]:
def compute_metrics(X: np.ndarray, labels: np.ndarray, tag: str) -> dict:
    """Compute Silhouette, Davies-Bouldin, Calinski-Harabasz."""
    unique = set(labels) - {-1}
    n_clusters = len(unique)
    n_noise    = int((labels == -1).sum())

    if n_clusters < 2:
        log.warning(f"  {tag}: only {n_clusters} cluster(s) — metrics N/A")
        return {"tag": tag, "n_clusters": n_clusters,
                "noise_frac": n_noise / len(labels),
                "silhouette": np.nan, "davies_bouldin": np.nan,
                "calinski_harabasz": np.nan}

    # Exclude noise points
    mask = labels != -1
    X_m, y_m = X[mask], labels[mask]

    # ── FIX: verificar que cada cluster tiene al menos 2 muestras ──────
    cluster_sizes = np.bincount(y_m)
    if np.any(cluster_sizes < 2):
        log.warning(f"  {tag}: cluster con 1 sola muestra — metrics N/A")
        return {"tag": tag, "n_clusters": n_clusters,
                "noise_frac": n_noise / len(labels),
                "silhouette": np.nan, "davies_bouldin": np.nan,
                "calinski_harabasz": np.nan}
    # ───────────────────────────────────────────────────────────────────

    sil = silhouette_score(X_m, y_m, sample_size=min(10_000, len(X_m)),
                           random_state=SEED)
    db  = davies_bouldin_score(X_m, y_m)
    ch  = calinski_harabasz_score(X_m, y_m)

    log.info(f"  {tag}: k={n_clusters}  noise={n_noise/len(labels):.1%}"
             f"  Sil={sil:.3f}  DB={db:.3f}  CH={ch:.1f}")
    return {"tag": tag, "n_clusters": n_clusters,
            "noise_frac": n_noise / len(labels),
            "silhouette": round(sil, 4),
            "davies_bouldin": round(db, 4),
            "calinski_harabasz": round(ch, 1)}

## 4. Algoritmo 1: K-Means

La función `run_kmeans` implementa el clustering con K-Means:
1.  **Barrido de `k`:** Se prueba un rango de valores para `k` (el número de clusters), de 2 a 12.
2.  **Curva del Codo (Elbow Curve):** Se grafica la inercia (suma de las distancias al cuadrado de las muestras a su centro de cluster más cercano) para cada `k`. El "codo" en la curva es un indicador del `k` óptimo.
3.  **Puntuación de Silueta:** Se grafica el coeficiente de silueta para cada `k`. El valor máximo de silueta es otro indicador del `k` óptimo.
4.  **Selección del mejor `k`:** Se elige el `k` que maximiza la puntuación de silueta.
5.  **Análisis de Estabilidad:** Se ejecuta K-Means con el `k` óptimo usando diferentes semillas aleatorias para evaluar la estabilidad de los resultados.
6.  **Resultados Finales:** Se devuelve el etiquetado de clusters y las métricas para el mejor modelo de K-Means.

In [5]:
def run_kmeans(X_pca: np.ndarray) -> tuple[np.ndarray, dict]:
    """
    Sweep k ∈ {2,..,12}, plot elbow + silhouette.
    Choose best k by Silhouette score.
    Repeat best k with 3 seeds and report mean ± std.
    """
    log.info("\n── K-Means ──────────────────────────────────────────────")
    K_RANGE = range(2, 13)
    inertias, silhouettes = [], []

    for k in K_RANGE:
        km = KMeans(n_clusters=k, init="k-means++", n_init=10,
                    random_state=SEED, max_iter=300)
        labels = km.fit_predict(X_pca)
        inertias.append(km.inertia_)
        sil = silhouette_score(X_pca, labels,
                               sample_size=min(10_000, len(X_pca)),
                               random_state=SEED)
        silhouettes.append(sil)
        log.info(f"  k={k:2d}  inertia={km.inertia_:,.0f}  sil={sil:.3f}")

    # ── Elbow + Silhouette figure ──────────────────────────────────────
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(list(K_RANGE), inertias, "bo-", ms=6)
    ax1.set_xlabel("k"); ax1.set_ylabel("Inertia")
    ax1.set_title("K-Means — Elbow Curve")
    ax1.grid(True, alpha=0.3)

    ax2.plot(list(K_RANGE), silhouettes, "rs-", ms=6)
    ax2.set_xlabel("k"); ax2.set_ylabel("Silhouette Score")
    ax2.set_title("K-Means — Silhouette vs k")
    ax2.grid(True, alpha=0.3)
    best_k = list(K_RANGE)[int(np.argmax(silhouettes))]
    ax2.axvline(best_k, color="green", ls="--", label=f"best k={best_k}")
    ax2.legend()
    fig.tight_layout()
    fig.savefig("figures/01_kmeans_elbow_silhouette.png", dpi=120)
    plt.close(fig)
    log.info(f"  → Best k by Silhouette: {best_k}")

    # ── Stability across seeds ─────────────────────────────────────────
    seed_results = []
    for s in [42, 123, 777]:
        km = KMeans(n_clusters=best_k, init="k-means++", n_init=10,
                    random_state=s, max_iter=300)
        labels_s = km.fit_predict(X_pca)
        m = compute_metrics(X_pca, labels_s, f"KMeans_k{best_k}_s{s}")
        seed_results.append(m)

    # Best labels = seed 42
    km_best = KMeans(n_clusters=best_k, init="k-means++", n_init=10,
                     random_state=42, max_iter=300)
    labels_km = km_best.fit_predict(X_pca)
    metrics_km = compute_metrics(X_pca, labels_km, f"KMeans_k{best_k}")

    # Stability summary
    for metric in ["silhouette", "davies_bouldin", "calinski_harabasz"]:
        vals = [r[metric] for r in seed_results]
        log.info(f"  KMeans {metric}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")

    return labels_km, metrics_km, inertias, silhouettes, best_k


## 5. Algoritmo 2: DBSCAN

La función `run_dbscan` implementa el clustering con DBSCAN:
1. **Estimación de `eps`:** Se grafica la distancia al k-NN más cercano. El codo se identifica usando el **percentil 90** de la distribución de distancias, que es la heurística estándar (no el percentil 2 original, que producía eps artificialmente bajos).
2. **Barrido de hiperparámetros:** Grilla sobre `eps` (8 valores alrededor del codo) × `min_samples` ∈ {5, 10, 2×d}.
3. **Criterio de selección del mejor modelo:** Se usa un **score compuesto** que balancea Silhouette, Davies-Bouldin normalizado y fracción de ruido, en lugar de maximizar Silhouette solo (que favorecía configuraciones con decenas de micro-clusters). Se filtra además `noise_frac > 0.40` y `n_clusters > 30` para evitar soluciones degeneradas.
4. **Análisis del ruido:** Se generan perfiles de los puntos clasificados como ruido (label = -1) para interpretación de dominio.


In [6]:
def run_dbscan(X_pca: np.ndarray) -> tuple[np.ndarray, dict]:
    """
    k-NN distance plot to choose eps (percentile-90 heuristic, not p2).
    Sweep eps x min_samples; select best config by composite score
    (balances Silhouette, Davies-Bouldin, and noise fraction).
    """
    log.info("\n── DBSCAN ───────────────────────────────────────────────")
    d = X_pca.shape[1]
    # min_samples candidates: 5, 10, 2*d (deduplicated)
    MIN_SAMPLES_LIST = sorted(set([5, 10, max(5, 2 * d)]))

    # ── k-NN distance plot ─────────────────────────────────────────────
    min_samples_plot = 5
    nn = NearestNeighbors(n_neighbors=min_samples_plot)
    nn.fit(X_pca)
    distances, _ = nn.kneighbors(X_pca)
    knn_dist = np.sort(distances[:, -1])

    # FIX: use 90th percentile as elbow heuristic (standard practice),
    # NOT 2nd percentile, which generated eps too small → hundreds of micro-clusters
    elbow_eps = float(np.percentile(knn_dist, 90))
    log.info(f"  k-NN 90th-pct distance (elbow eps): {elbow_eps:.4f}")

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(knn_dist, lw=1)
    ax.set_xlabel("Points sorted by distance")
    ax.set_ylabel(f"{min_samples_plot}-NN distance")
    ax.set_title("DBSCAN — k-NN Distance Plot (eps guidance)")
    ax.grid(True, alpha=0.3)
    ax.axhline(elbow_eps, color="red", ls="--",
               label=f"eps≈{elbow_eps:.3f} (90th percentile)")
    ax.legend()
    fig.tight_layout()
    fig.savefig("figures/02_dbscan_knn_distance.png", dpi=120)
    plt.close(fig)

    # ── Sweep eps × min_samples ────────────────────────────────────────
    # Range: 0.5× to 2× the elbow, 8 steps
    eps_candidates = np.round(
        np.linspace(max(0.05, elbow_eps * 0.5), elbow_eps * 2.0, 8), 3
    )
    log.info(f"  eps candidates: {eps_candidates}")

    # FIX: composite score = Silhouette (higher better) + inv(DB) (lower DB better)
    # penalise high noise. Only considers configs with 2..30 clusters and noise<40%.
    best_score  = -np.inf
    best_labels = None
    best_cfg    = {}
    sweep_rows  = []

    for ms in MIN_SAMPLES_LIST:
        for eps in eps_candidates:
            db_model = DBSCAN(eps=eps, min_samples=int(ms))
            lbl      = db_model.fit_predict(X_pca)
            n_cl     = len(set(lbl) - {-1})
            noise    = (lbl == -1).mean()

            # Skip degenerate configs: no meaningful clustering or too much noise
            if n_cl < 2 or n_cl > 30 or noise > 0.40:
                row = {"eps": eps, "min_samples": ms,
                       "n_clusters": n_cl, "noise_frac": round(noise, 3),
                       "silhouette": np.nan, "davies_bouldin": np.nan,
                       "calinski_harabasz": np.nan, "tag": np.nan}
                sweep_rows.append(row)
                continue

            m = compute_metrics(X_pca, lbl, f"DBSCAN(eps={eps},ms={ms})")
            sweep_rows.append({**m, "eps": eps, "min_samples": ms})

            if np.isnan(m["silhouette"]):
                continue

            # Composite score: Silhouette - 0.3*Davies_Bouldin - 0.5*noise_frac
            # (all components in comparable ranges after normalisation)
            composite = (m["silhouette"]
                         - 0.3 * m["davies_bouldin"]
                         - 0.5 * noise)
            log.info(f"    eps={eps:.3f} ms={ms:2d} → k={n_cl} "
                     f"noise={noise:.2f} Sil={m['silhouette']:.3f} "
                     f"composite={composite:.3f}")

            if composite > best_score:
                best_score  = composite
                best_labels = lbl.copy()
                best_cfg    = {"eps": eps, "min_samples": int(ms)}

    pd.DataFrame(sweep_rows).to_csv("results/dbscan_sweep.csv", index=False)

    if best_labels is None:
        log.warning("  DBSCAN: no valid configuration found; "
                    "using fallback eps=elbow, min_samples=5")
        best_cfg = {"eps": round(elbow_eps, 3), "min_samples": 5}
        best_labels = DBSCAN(**best_cfg).fit_predict(X_pca)

    log.info(f"  → Best DBSCAN config: {best_cfg} | composite={best_score:.3f}")
    metrics_db = compute_metrics(X_pca, best_labels, "DBSCAN_best")

    # ── Noise-point profile ────────────────────────────────────────────
    # FIX: analyse noise points in domain terms (required by challenge)
    noise_mask = best_labels == -1
    log.info(f"  Noise points: {noise_mask.sum()} ({noise_mask.mean():.1%})")

    return best_labels, metrics_db, best_cfg, noise_mask


## 6. Algoritmo 3: Clustering Jerárquico Aglomerativo

La función `run_hierarchical` implementa el clustering jerárquico:
1.  **Dendrograma:** Se genera un dendrograma en una submuestra de los datos para visualizar la estructura jerárquica de los clusters. El método de enlace (`linkage`) utilizado aquí es "ward", que tiende a encontrar clusters de tamaño similar.
2.  **Comparación de Métodos de Enlace:** Se comparan tres métodos de enlace (`ward`, `complete`, `average`) para agrupar los datos. El número de clusters se fija en `best_k` (el valor óptimo encontrado por K-Means) para permitir una comparación directa.
3.  **Selección del Mejor Enlace:** Se elige el método de enlace que resulta en la puntuación de silueta más alta.
4.  **Resultados Finales:** Se devuelven las etiquetas de los clusters y las métricas para el mejor modelo jerárquico.

In [7]:
def run_hierarchical(X_pca: np.ndarray, best_k: int) -> tuple[np.ndarray, dict]:
    """
    Ward, complete, and average linkage comparison.
    Dendrogram plotted on a 2 000-sample subset.
    FIX: sweeps k ∈ {2,3,4} independently from K-Means to validate
    the dendrogram-suggested cut, then reports the best (k, linkage) pair.
    """
    log.info("\n── Hierarchical Clustering ──────────────────────────────")

    # ── Dendrogram on sample ───────────────────────────────────────────
    rng = np.random.RandomState(SEED)
    sample_idx = rng.choice(len(X_pca), size=min(2000, len(X_pca)), replace=False)
    X_sample   = X_pca[sample_idx]
    Z          = linkage(X_sample, method="ward")

    fig, ax = plt.subplots(figsize=(14, 5))
    dendrogram(Z, ax=ax, truncate_mode="level", p=6,
               color_threshold=0.7 * max(Z[:, 2]))
    ax.set_title("Hierarchical Clustering — Dendrogram (Ward, 2 000-sample)")
    ax.set_xlabel("Sample index / cluster size")
    ax.set_ylabel("Distance")
    fig.tight_layout()
    fig.savefig("figures/03_hierarchical_dendrogram.png", dpi=120)
    plt.close(fig)

    # ── FIX: sweep k ∈ {2,3,4} across all linkages independently ──────
    # K-Means found best_k, but dendrogram should validate it.
    # Also test k+1 and k+2 to check for richer structure.
    K_OPTIONS = sorted(set([2, best_k, min(best_k + 1, 5), min(best_k + 2, 6)]))
    best_sil, best_labels, best_linkage, best_hc_k = -2, None, "ward", best_k

    hc_sweep_rows = []
    for link in ["ward", "complete", "average"]:
        for k_hc in K_OPTIONS:
            hc  = AgglomerativeClustering(n_clusters=k_hc, linkage=link)
            lbl = hc.fit_predict(X_pca)
            m   = compute_metrics(X_pca, lbl, f"HC_{link}_k{k_hc}")
            hc_sweep_rows.append({**m, "linkage": link, "k": k_hc})
            if not np.isnan(m["silhouette"]) and m["silhouette"] > best_sil:
                best_sil      = m["silhouette"]
                best_labels   = lbl.copy()
                best_linkage  = link
                best_hc_k     = k_hc

    pd.DataFrame(hc_sweep_rows).to_csv("results/hierarchical_sweep.csv", index=False)
    log.info(f"  → Best HC: linkage={best_linkage}, k={best_hc_k}, "
             f"Sil={best_sil:.4f}")

    metrics_hc = compute_metrics(X_pca, best_labels,
                                 f"HC_{best_linkage}_k{best_hc_k}")
    return best_labels, metrics_hc, best_linkage, best_hc_k


## 7. Visualización de Clusters en 2D

La función `plot_clusters_2d` se utiliza para visualizar los resultados de cada algoritmo.
1.  **Reducción a 2D:** Aplica PCA nuevamente para reducir los datos a solo dos componentes principales, que se pueden graficar en un plano.
2.  **Gráfico de Dispersión:** Crea un gráfico de dispersión donde cada punto es una observación (`estado × mes`) y su color corresponde a la etiqueta del cluster asignada.
3.  **Etiquetas y Leyenda:** Se añaden etiquetas a los ejes que muestran el porcentaje de varianza explicado por cada componente principal y una leyenda para identificar los clusters.

Esta visualización proporciona una intuición visual de la separación y la forma de los clusters.

In [8]:
def plot_clusters_2d(X_pca: np.ndarray, labels: np.ndarray,
                     title: str, fname: str,
                     pca2_obj: PCA | None = None) -> None:
    """
    2-D scatter coloured by cluster label.

    FIX: instead of fitting a new PCA on top of X_pca (which is already
    reduced), we use the first two components of X_pca directly (PC1, PC2).
    If the caller passes a pca2_obj with explained_variance_ratio_, those
    percentages are shown on the axes.
    """
    # Use PC1 and PC2 directly — X_pca[:,0] and X_pca[:,1]
    X2 = X_pca[:, :2]

    if pca2_obj is not None:
        var1 = pca2_obj.explained_variance_ratio_[0] * 100
        var2 = pca2_obj.explained_variance_ratio_[1] * 100
    else:
        var1 = var2 = None

    unique = sorted(set(labels))
    # Use tab20 for ≤20 clusters, fall back to a continuous map for more
    if len(unique) <= 20:
        cmap = cm.get_cmap("tab20", max(len(unique), 2))
    else:
        cmap = cm.get_cmap("nipy_spectral", len(unique))

    fig, ax = plt.subplots(figsize=(9, 6))
    for i, lbl in enumerate(unique):
        mask  = labels == lbl
        color = "grey" if lbl == -1 else cmap(i)
        name  = "Noise" if lbl == -1 else f"Cluster {lbl}"
        ax.scatter(X2[mask, 0], X2[mask, 1],
                   c=[color], s=3, alpha=0.35, label=name, rasterized=True)

    xlabel = f"PC1 ({var1:.1f}%)" if var1 is not None else "PC1"
    ylabel = f"PC2 ({var2:.1f}%)" if var2 is not None else "PC2"
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(loc="best", markerscale=4, fontsize=7,
              ncol=max(1, len(unique) // 8))
    fig.tight_layout()
    fig.savefig(f"figures/{fname}", dpi=120)
    plt.close(fig)
    log.info(f"  Saved: figures/{fname}")


## 8. Perfilado de Clusters

La función `cluster_profiles` calcula el perfil de cada cluster.
1.  **Agrupación por Cluster:** Agrupa los datos originales por la etiqueta de cluster asignada.
2.  **Cálculo de la Media:** Para cada cluster, calcula el valor medio de cada una de las características originales.
3.  **Guardado en CSV:** Guarda estos perfiles en un archivo CSV.

El perfilado es crucial para la interpretación. Nos permite entender qué caracteriza a cada grupo. Por ejemplo, un cluster podría tener en promedio "ventas altas y precios bajos", mientras que otro podría representar "ventas bajas con alta volatilidad".

In [9]:
def cluster_profiles(df: pd.DataFrame, labels: np.ndarray,
                     features: list[str], tag: str,
                     include_noise: bool = False) -> pd.DataFrame:
    """
    Compute per-cluster mean of each feature and save to CSV.

    FIX: include_noise=True adds a row for noise points (label == -1)
    so the caller can inspect their domain characteristics.
    """
    df_tmp = df.copy()
    df_tmp["cluster"] = labels

    # Clusters ≥ 0
    profile = (df_tmp[df_tmp["cluster"] >= 0]
               .groupby("cluster")[features]
               .mean()
               .round(3))
    profile["n_samples"] = (
        df_tmp[df_tmp["cluster"] >= 0]
        .groupby("cluster")
        .size()
    )

    if include_noise and (labels == -1).any():
        noise_row = df_tmp[df_tmp["cluster"] == -1][features].mean().round(3)
        noise_row["n_samples"] = int((labels == -1).sum())
        noise_df = pd.DataFrame(noise_row).T
        noise_df.index = [-1]
        profile = pd.concat([noise_df, profile])

    profile.to_csv(f"results/{tag}_profiles.csv")
    log.info(f"  Cluster profiles saved: results/{tag}_profiles.csv")
    return profile


## 9. Tabla Comparativa de Resultados

La función `save_comparison` consolida las métricas de evaluación de los tres algoritmos en una única tabla.
1.  **Creación de DataFrame:** Convierte la lista de diccionarios de métricas en un DataFrame de pandas.
2.  **Guardado y Visualización:** Guarda la tabla en un archivo CSV y la imprime en la consola.

Esta tabla permite una comparación rápida y directa del rendimiento de K-Means, DBSCAN y el clustering jerárquico según las métricas seleccionadas.

In [10]:
def save_comparison(metrics_list: list[dict], fname: str = "comparison_table.csv") -> None:
    """Save comparison table. Accepts any number of rows (including ablation runs)."""
    df = pd.DataFrame(metrics_list)
    cols = ["tag", "n_clusters", "noise_frac",
            "silhouette", "davies_bouldin", "calinski_harabasz"]
    df = df[[c for c in cols if c in df.columns]]
    df.to_csv(f"results/{fname}", index=False)

    log.info("\n" + "="*80)
    log.info("  COMPARISON TABLE")
    log.info("="*80)
    log.info(df.to_string(index=False))
    log.info("="*80)


## 10. Bloque Principal de Ejecución (`main`)

El `main` orquesta el flujo de trabajo completo:
1. `load_and_preprocess` → datos listos.
2. K-Means → sweep k, estabilidad, visualización, perfiles.
3. DBSCAN (corregido) → sweep con criterio compuesto, análisis de ruido.
4. Hierarchical (corregido) → sweep k×linkage independiente del K-Means.
5. Tabla comparativa de los tres algoritmos.
6. **Ablación de features** (corregida):
   - Subset temporal (`month_sin/cos`, `year_trend`, `roll12_*`, `yoy_growth`).
   - Subset económico (`price`, `revenue_per_mwh`, `yoy_growth`).
   - Cada subset tiene su propio pipeline (escala + PCA separados), visualización 2-D y perfiles.
   - Resultados incluidos en la tabla comparativa final.


In [11]:
def main():
    log.info("Challenge 5 — EIA Energy Clustering")
    log.info("="*60)

    # 1. Load and preprocess (full feature set)
    X_scaled, X_pca, df, FEATURES, pca = load_and_preprocess()

    # 2. K-Means
    labels_km, metrics_km, inertias, silhouettes, best_k = run_kmeans(X_pca)
    plot_clusters_2d(X_pca, labels_km,
                     f"K-Means (k={best_k}) — PCA Projection",
                     "04_kmeans_clusters.png",
                     pca2_obj=pca)
    profiles_km = cluster_profiles(df, labels_km, FEATURES, "kmeans")

    # 3. DBSCAN — now returns noise_mask for domain profiling
    labels_db, metrics_db, best_cfg_db, noise_mask_db = run_dbscan(X_pca)
    plot_clusters_2d(X_pca, labels_db,
                     f"DBSCAN (eps={best_cfg_db['eps']},"
                     f" min_samples={best_cfg_db['min_samples']}) — PCA",
                     "05_dbscan_clusters.png",
                     pca2_obj=pca)
    # FIX: include noise profile for domain interpretation
    profiles_db = cluster_profiles(df, labels_db, FEATURES, "dbscan",
                                   include_noise=True)

    # 4. Hierarchical — now sweeps k independently from K-Means
    labels_hc, metrics_hc, best_link, best_hc_k = run_hierarchical(X_pca, best_k)
    plot_clusters_2d(X_pca, labels_hc,
                     f"Hierarchical ({best_link}, k={best_hc_k}) — PCA",
                     "06_hierarchical_clusters.png",
                     pca2_obj=pca)
    profiles_hc = cluster_profiles(df, labels_hc, FEATURES, "hierarchical")

    # 5. Comparison table (main algorithms only)
    save_comparison([metrics_km, metrics_db, metrics_hc])

    # ── 6. Feature ablation ────────────────────────────────────────────
    # FIX: two domain-motivated subsets, each with its own scale+PCA pipeline.
    # Original code incorrectly sliced X_pca[:,: N] instead of re-fitting PCA
    # on the raw subset, and only defined one subset without visualisation.

    log.info("\n── Feature Ablation ──────────────────────────────────────")

    SUBSET_TEMPORAL  = ["month_sin", "month_cos", "year_trend",
                        "roll12_mean", "roll12_std", "yoy_growth"]
    SUBSET_ECONOMIC  = ["price", "revenue_per_mwh", "yoy_growth"]

    ablation_metrics = [metrics_km]  # baseline: full features
    ablation_metrics[0] = {**metrics_km, "tag": "KMeans_full_features"}

    for subset_name, subset_feats in [("temporal",  SUBSET_TEMPORAL),
                                       ("economic",  SUBSET_ECONOMIC)]:
        log.info(f"  Subset: {subset_name} → {subset_feats}")

        # Independent scale + PCA for this subset
        X_sub_raw    = StandardScaler().fit_transform(df[subset_feats].values)
        n_comp_sub   = min(len(subset_feats), 4)
        pca_sub      = PCA(n_components=n_comp_sub, random_state=SEED)
        X_sub_pca    = pca_sub.fit_transform(X_sub_raw)

        km_sub  = KMeans(n_clusters=best_k, init="k-means++", n_init=10,
                         random_state=SEED, max_iter=300)
        lbl_sub = km_sub.fit_predict(X_sub_pca)
        m_sub   = compute_metrics(X_sub_pca, lbl_sub,
                                  f"KMeans_{subset_name}_subset")

        # 2-D visualisation of subset clustering
        plot_clusters_2d(X_sub_pca, lbl_sub,
                         f"K-Means ({subset_name} features, k={best_k}) — PCA",
                         f"07_kmeans_{subset_name}_subset.png",
                         pca2_obj=pca_sub)

        # Profiles of subset clusters
        cluster_profiles(df, lbl_sub, subset_feats,
                         f"kmeans_{subset_name}_subset")

        ablation_metrics.append(m_sub)
        log.info(f"    Full Sil={metrics_km['silhouette']:.3f} | "
                 f"{subset_name} Sil={m_sub['silhouette']:.3f}")

    # Save extended comparison table (includes ablation runs)
    save_comparison(ablation_metrics + [metrics_db, metrics_hc],
                    fname="comparison_table_full.csv")

    log.info("\n✓ Challenge 5 complete.")
    log.info("  Figures : figures/")
    log.info("  Results : results/comparison_table.csv")
    log.info("            results/comparison_table_full.csv  (includes ablation)")
    log.info("            results/dbscan_sweep.csv")
    log.info("            results/hierarchical_sweep.csv")
    log.info("  Logs    : logs/challenge5.log")


if __name__ == "__main__":
    main()


2026-05-16 19:32:22,924 | INFO | Challenge 5 — EIA Energy Clustering
2026-05-16 19:32:22,932 | INFO | ============================================================
2026-05-16 19:32:22,932 | INFO | Loading data from eia_retail_sales.csv


2026-05-16 19:32:23,287 | INFO |   Raw rows: 111,972
2026-05-16 19:32:23,380 | INFO |   After ALL-sector filter: 18,662 rows | 62 states | 2001-01-01 → 2026-01-01
2026-05-16 19:32:23,474 | INFO |   Final dataset: 17,918 rows × 8 features
2026-05-16 19:32:23,491 | INFO |   PCA: 5 components retain 90.8% variance
2026-05-16 19:32:24,196 | INFO | 
── K-Means ──────────────────────────────────────────────
2026-05-16 19:32:28,732 | INFO |   k= 2  inertia=98,999  sil=0.716
2026-05-16 19:32:31,654 | INFO |   k= 3  inertia=75,878  sil=0.302
2026-05-16 19:32:35,397 | INFO |   k= 4  inertia=63,113  sil=0.224
2026-05-16 19:32:38,817 | INFO |   k= 5  inertia=54,215  sil=0.225
2026-05-16 19:32:42,442 | INFO |   k= 6  inertia=49,709  sil=0.217
2026-05-16 19:32:45,706 | INFO |   k= 7  inertia=45,869  sil=0.209
2026-05-16 19:32:48,867 | INFO |   k= 8  inertia=42,584  sil=0.205
2026-05-16 19:32:52,005 | INFO |   k= 9  inertia=39,767  sil=0.208
2026-05-16 19:32:55,358 | INFO |   k=10  inertia=37,096  si